In [2]:
import numpy as np

In [53]:
class Neural:
  def __init__(self,input_size=784,hidden_layers=[128,128],output_size=10):
    self.input_size = input_size
    self.hidden_layers = hidden_layers
    self.output_size = output_size
    self.weights = []
    self.biases = []
    self.layers = [] # Post-ReLU values
    self.layers_z = [] #Pre-ReLU values

    #Input for hidden layers
    self.weights.append(0.1 * np.random.randn(input_size,hidden_layers[0]))
    self.biases.append(np.zeros((1,hidden_layers[0])))

    #Hidden Layers Network
    for i in range(len(hidden_layers)-1):
      self.weights.append(0.1 * np.random.randn(hidden_layers[i],hidden_layers[i+1]))
      self.biases.append(np.zeros((1,hidden_layers[i+1])))

    #Hidden layers to Output
    self.weights.append(0.1 * np.random.randn(hidden_layers[-1],output_size))
    self.biases.append(np.zeros((1,output_size)))

  def forward(self,inputs):
    self.layers = [inputs]
    self.layers_z = []
    #Dot product of inputs and weights
    for i in range(len(self.weights)):
      if i == (len(self.weights)-1):
        self.layers_z.append(np.dot(self.layers[-1],self.weights[i])+self.biases[i])
      else: 
        output = np.dot(self.layers[-1],self.weights[i])+self.biases[i]
        self.layers_z.append(output)
        output = self.relu(output)
        self.layers.append(output)
    self.layers.append(self.softmax_act(self.layers_z[-1]))
    return self.layers[-1]

  def relu(self,inputs):
    return np.where(inputs < 0, 0, inputs)

  def softmax_act(self,inputs):
    stable_input = inputs - np.max(inputs,axis=1,keepdims=True)
    exp_values = np.exp(stable_input)
    return exp_values / np.sum(exp_values,axis=1,keepdims=True)

  def loss(self,input,target,epsilon=1e-15):
    # Clip predictions to avoid numerical instability (log(0))
    y_pred = np.clip(input, epsilon, 1 - epsilon)
    
    # Calculate loss for each sample and then the average over the batch
    loss = -np.mean(np.sum(target * np.log(y_pred), axis=-1))
    return loss
  
  def backpass(self,inputs,target,batch_size):
    dOutput = np.subtract(self.layers[-1],target)
    dgradient = dOutput/batch_size
    d_weights_list = []
    d_biases_list = []
    for i in range(len(self.weights)-1,-1,-1):
      dweights = np.dot(self.layers[i].T, dgradient)
      dbiases = np.sum(dgradient,axis=0)
      dgradient = np.dot(dgradient, self.weights[i].T)
      if i != (len(self.weights)-1) and i!=0:
        dgradient = np.multiply(dgradient,self.layers_z[i-1]>0)
      d_weights_list.append(dweights)
      d_biases_list.append(dbiases)
    d_weights_list = d_weights_list[::-1]
    d_biases_list = d_biases_list[::-1]
    return d_weights_list, d_biases_list

  def update(self, d_weights_list, d_biases_list, learning_rate=0.01):
    for i in range(len(self.weights)):
        self.weights[i] -= learning_rate * d_weights_list[i]
        self.biases[i]  -= learning_rate * d_biases_list[i]

  def train(self,inputs,target,batch_size,epochs,learning_rate):
    for i in range(epochs):
      print(f"Epoch {i}")
      indices = np.random.permutation(len(inputs))
      r_inputs = inputs[indices]
      r_target = target[indices]
      losses = []
      for j in range(0,len(inputs),batch_size):
        batch_inputs = r_inputs[j:j+batch_size]
        batch_target = r_target[j:j+batch_size]
        f_pass = self.forward(batch_inputs)
        loss = self.loss(f_pass,batch_target)
        losses.append(loss)
        d_w , d_l = self.backpass(batch_inputs,batch_target,batch_size)
        self.update(d_w,d_l,learning_rate)
      print(f"Loss : {np.mean(losses)}")


In [82]:
def true_vals(array):
  array = array.flatten()
  true_values = np.zeros((array.size,array.max()+1))
  true_values[np.arange(array.size),array] = 1
  return true_values

In [83]:
def accuracy(y_pred,y_true):
    pred = np.argmax(y_pred,axis=1)
    true = np.argmax(y_true,axis=1)
    count = (pred == true).sum()
    return count/len(y_true)

In [4]:
img_data = np.load("Datasets/train_images.npy")
img_labels = np.load("Datasets/train_labels.npy")

In [84]:
img_data = np.reshape(img_data,(60000,784))
img_data = img_data/255
img_labels = np.reshape(img_labels,(60000,1))
img_labels = true_vals(img_labels)


In [90]:
neural_net = Neural()
neural_net.train(img_data, img_labels, batch_size=128, epochs=100, learning_rate=0.1)

Epoch 0
Loss : 0.45110128322348564
Epoch 1
Loss : 0.2238524718107294
Epoch 2
Loss : 0.17224484560897335
Epoch 3
Loss : 0.1399259208783475
Epoch 4
Loss : 0.11814316136001403
Epoch 5
Loss : 0.10044616124725668
Epoch 6
Loss : 0.08874157530476195
Epoch 7
Loss : 0.07824865670570191
Epoch 8
Loss : 0.07057581049419052
Epoch 9
Loss : 0.06303933230527138
Epoch 10
Loss : 0.056926197431728616
Epoch 11
Loss : 0.051534585548175665
Epoch 12
Loss : 0.04573021486227803
Epoch 13
Loss : 0.041287786852976696
Epoch 14
Loss : 0.03765784366266301
Epoch 15
Loss : 0.03414315995977748
Epoch 16
Loss : 0.029956430056311118
Epoch 17
Loss : 0.027371528648000406
Epoch 18
Loss : 0.024553206317710248
Epoch 19
Loss : 0.02246307672152487
Epoch 20
Loss : 0.01942275885220577
Epoch 21
Loss : 0.017092304153883384
Epoch 22
Loss : 0.015701023857089276
Epoch 23
Loss : 0.014692443809591732
Epoch 24
Loss : 0.012407291184380399
Epoch 25
Loss : 0.01088752584731847
Epoch 26
Loss : 0.009821724601617416
Epoch 27
Loss : 0.00854749596

In [86]:
test_data = np.load("Datasets/test_images.npy")
test_img_labels = np.load("Datasets/test_labels.npy")

In [87]:
test_data = np.reshape(test_data,(len(test_data),784))
test_labels = np.reshape(test_img_labels,(len(test_img_labels),1))
test_labels = true_vals(test_labels)


In [91]:
result = neural_net.forward(test_data)

In [92]:
print("Accuracy :",accuracy(result,test_labels))

Accuracy : 0.9746
